In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, shutil, sqlite3
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/Goszakup_agent"
DB_DRIVE = f"{SAVE_DIR}/goszakup_raw.db"
DB_LOCAL = "/content/goszakup_raw.db"

# Drive-тағы файлды уақытша дискіге көшіреміз: сұраныстар тезірек жүреді
print("Drive-тағы файл:", round(os.path.getsize(DB_DRIVE) / 1e6, 1), "МБ")
shutil.copy(DB_DRIVE, DB_LOCAL)

db = sqlite3.connect(DB_LOCAL)
q = lambda sql: pd.read_sql_query(sql, db)

BINS = """
000740001307 020240002363 020440003656 030440003698 050740004819
051040005150 100140011059 120940001946 140340016539 150540000186
171041003124 210240019348 210240033968 210941010761 230740013340
231040023028 780140000023 900640000128 940740000911 940940000384
960440000220 970940001378 971040001050 980440001034 981140001551
990340005977 990740002243
""".split()

print(q("SELECT entity, COUNT(*) AS n FROM raw GROUP BY entity ORDER BY n DESC"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive-тағы файл: 737.7 МБ
          entity       n
0            lot  116368
1  contract_unit  105166
2           plan   74100
3   trd_buy_skip   38045
4       contract   37039
5        trd_buy   36488
6           done      27


In [ ]:
import pandas as pd
q = lambda sql: pd.read_sql_query(sql, db)

print(q("SELECT entity, COUNT(*) AS n FROM raw GROUP BY entity ORDER BY n DESC"))

done = {r[0] for r in db.execute("SELECT obj_id FROM raw WHERE entity='done'")}
print("\nАяқталмаған ұйымдар:", [b for b in BINS if b not in done] or "жоқ")

          entity       n
0            lot  116368
1  contract_unit  105166
2           plan   74100
3   trd_buy_skip   38045
4       contract   37039
5        trd_buy   36488
6           done      27

Аяқталмаған ұйымдар: жоқ


In [ ]:
by_bin = q("""
SELECT bin,
       SUM(entity='plan')          AS plan,
       SUM(entity='lot')           AS lot,
       SUM(entity='trd_buy')       AS trd_buy,
       SUM(entity='contract')      AS contract,
       SUM(entity='contract_unit') AS units
FROM raw
WHERE entity IN ('plan','lot','trd_buy','contract','contract_unit')
GROUP BY bin ORDER BY contract DESC
""")
print(by_bin.to_string())

             bin   plan    lot  trd_buy  contract  units
0   990740002243  16136  28572     5859      7721  24926
1   231040023028   6454  12466     2972      3256   9673
2   051040005150   3934   8052     3834      2964   5095
3   030440003698   5692   5704     2100      2340  12401
4   971040001050   4961   9540     1547      2286   5668
5   020440003656   6412   4677     2703      2146   9934
6   171041003124   3759   7368     1964      2063   4124
7   100140011059   6054  12210     2187      1800   5189
8   780140000023   2199   5401     1611      1579   2953
9   970940001378   3586   2747     1199      1325   6012
10  981140001551   1772   1252      877      1305   2151
11  210240033968   1118   2668     1275      1252   1742
12  980440001034   1793   1743     1446      1163   3315
13  940740000911   1642   2879      908      1008   1685
14  210941010761   2921   2384     1033       862   3273
15  230740013340    809    976      822       670   1145
16  940940000384    731   1350 

In [ ]:
print(q("""
SELECT entity, year, COUNT(*) AS n FROM raw
WHERE entity IN ('plan','trd_buy','contract')
GROUP BY entity, year ORDER BY entity, year
""").pivot(index="entity", columns="year", values="n"))

year       2024   2025   2026
entity                       
contract  14025  13035   9979
plan      27631  26060  20409
trd_buy   11464  14453  10571


In [ ]:
print(q("""
SELECT COUNT(*) AS contracts_without_units
FROM raw
WHERE entity = 'contract'
  AND CAST(obj_id AS INTEGER) NOT IN (
      SELECT json_extract(payload, '$.contract_registry_id')
      FROM raw
      WHERE entity = 'contract_unit'
        AND json_extract(payload, '$.contract_registry_id') IS NOT NULL
  )
"""))

   contracts_without_units
0                        2


In [ ]:
# Шарттың тауар жолдары жоспар тармақтарына қаншалықты қосылады
print(q("""
SELECT COUNT(*) AS units_total,
       SUM(p.obj_id IS NULL) AS plan_not_found
FROM raw u
LEFT JOIN raw p
  ON p.entity='plan'
 AND p.obj_id = CAST(json_extract(u.payload,'$.pln_point_id') AS TEXT)
WHERE u.entity='contract_unit'
"""))

# Тауар жолдары жоқ шарттар
print(q("""
SELECT COUNT(*) AS contracts_without_units FROM raw c
WHERE c.entity='contract' AND NOT EXISTS (
  SELECT 1 FROM raw u WHERE u.entity='contract_unit'
   AND json_extract(u.payload,'$.contract_registry_id') = CAST(c.obj_id AS INTEGER))
"""))

# Бір көзден сатып алу (хабарландыруы жоқ шарттар)
print(q("""
SELECT SUM(json_extract(payload,'$.trd_buy_id') = 0) AS no_tender,
       COUNT(*) AS total
FROM raw WHERE entity='contract'
"""))

# Бағасы нөл немесе жоқ тауар жолдары
print(q("""
SELECT COUNT(*) AS bad_price FROM raw
WHERE entity='contract_unit'
  AND (json_extract(payload,'$.item_price') IS NULL
       OR json_extract(payload,'$.item_price') <= 0)
"""))

   units_total  plan_not_found
0       105166           15843


In [ ]:
db.execute("DROP TABLE IF EXISTS miss")
db.execute("""
CREATE TEMP TABLE miss AS
SELECT u.bin, u.year,
       json_extract(u.payload,'$.pln_point_id') AS pid,
       json_extract(u.payload,'$.lot_id')       AS lot_id
FROM raw u
LEFT JOIN raw p
  ON p.entity='plan'
 AND p.obj_id = CAST(json_extract(u.payload,'$.pln_point_id') AS TEXT)
WHERE u.entity='contract_unit' AND p.obj_id IS NULL
""")

# 1) pln_point_id бос па? Лоты бар ма?
print(q("""
SELECT COUNT(*) AS missing,
       SUM(pid IS NULL OR pid = 0)       AS empty_pid,
       SUM(lot_id IS NULL OR lot_id = 0) AS empty_lot
FROM miss
"""))

# 2) Жылдар бойынша: қай жылда үлесі жоғары
print(q("""
SELECT u.year,
       COUNT(*) AS units,
       (SELECT COUNT(*) FROM miss m WHERE m.year = u.year) AS missing
FROM raw u WHERE u.entity='contract_unit'
GROUP BY u.year
"""))

# 3) Ұйымдар бойынша: ең көп жоғалған 10 ұйым
print(q("""
SELECT m.bin, COUNT(*) AS missing,
       ROUND(100.0 * COUNT(*) /
         (SELECT COUNT(*) FROM raw u WHERE u.entity='contract_unit' AND u.bin=m.bin), 1) AS pct
FROM miss m GROUP BY m.bin ORDER BY missing DESC LIMIT 10
"""))

# # 4) Лот арқылы жоспар тармағын табуға бола ма?
# print(q("""
# SELECT COUNT(*) AS missing,
#        SUM(l.obj_id IS NOT NULL) AS lot_found,
#        SUM(EXISTS (
#            SELECT 1 FROM json_each(l.payload, '$.point_list') j
#            JOIN raw p ON p.entity='plan' AND p.obj_id = CAST(j.value AS TEXT)
#        )) AS plan_found_via_lot
# FROM miss m
# LEFT JOIN raw l ON l.entity='lot' AND l.obj_id = CAST(m.lot_id AS TEXT)
# """))

# 5) Кейін API арқылы тексеру үшін бірнеше мысал ID
print(q("SELECT pid, lot_id, bin, year FROM miss WHERE pid > 0 LIMIT 5"))

   missing  empty_pid  empty_lot
0    15843          0        981
   year  units  missing
0  2024  42061     4906
1  2025  35641     3851
2  2026  27464     7086
            bin  missing   pct
0  990740002243     3618  14.5
1  020440003656     2611  26.3
2  970940001378     1609  26.8
3  030440003698     1502  12.1
4  231040023028     1208  12.5
5  980440001034      983  29.7
6  051040005150      811  15.9
7  171041003124      627  15.2
8  971040001050      454   8.0
9  210240033968      387  22.2
        pid    lot_id           bin  year
0  64427101  28554232  020440003656  2024
1  64442332  28549087  020440003656  2024
2  64442335  28549087  020440003656  2024
3  64442326  28549087  020440003656  2024
4  64442341  28549087  020440003656  2024


In [ ]:
# 4) Лот арқылы жоспар тармағын табуға бола ма?
# Жоспар тармақтарының барлық ID-лері (бір рет жүктейміз)
plan_ids = {r[0] for r in db.execute("SELECT obj_id FROM raw WHERE entity='plan'")}

# Жоғалған жолдарға қатысты лоттардың point_list тізімдері
lot_points = {}
for (lid,) in db.execute("SELECT DISTINCT lot_id FROM miss WHERE lot_id > 0").fetchall():
    row = db.execute("SELECT payload FROM raw WHERE entity='lot' AND obj_id=?",
                     (str(lid),)).fetchone()
    if row:
        lot_points[lid] = json.loads(row[0]).get("point_list") or []

# Санау
missing = lot_found = plan_found_via_lot = 0
for (lid,) in db.execute("SELECT lot_id FROM miss").fetchall():
    missing += 1
    if lid in lot_points:
        lot_found += 1
        if any(str(p) in plan_ids for p in lot_points[lid]):
            plan_found_via_lot += 1

print("Жоғалған жолдар:", missing)
print("Лоты табылғаны:", lot_found)
print("Лот арқылы жоспар тармағы табылғаны:", plan_found_via_lot)

Жоғалған жолдар: 15843
Лоты табылғаны: 14308
Лот арқылы жоспар тармағы табылғаны: 2794


In [ ]:
miss_rows = db.execute("SELECT pid, lot_id FROM miss").fetchall()

# Лот арқылы да шешілмеген жолдардың жоспар ID-лері
unresolved_pids = set()
for pid, lid in miss_rows:
    pts = lot_points.get(lid, [])
    if not any(str(p) in plan_ids for p in pts):
        unresolved_pids.add(pid)
print("Жетіспейтін бірегей жоспар тармағы:", len(unresolved_pids))

# Лоттарда ЕНС ТРУ коды толтырылған ба (қосалқы дереккөз ретінде)
lot_enstru = {}
for lid in lot_points:
    row = db.execute("SELECT payload FROM raw WHERE entity='lot' AND obj_id=?",
                     (str(lid),)).fetchone()
    lot_enstru[lid] = [e for e in (json.loads(row[0]).get("enstru_list") or []) if e]
print("Лотында ЕНС ТРУ бар жоғалған жолдар:",
      sum(1 for pid, lid in miss_rows if lot_enstru.get(lid)))

Жетіспейтін бірегей жоспар тармағы: 10176
Лотында ЕНС ТРУ бар жоғалған жолдар: 1168


In [ ]:
import time, requests
from google.colab import userdata

TOKEN = userdata.get("GOSZAKUP_TOKEN")
BASE_URL = "https://ows.goszakup.gov.kz"

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {TOKEN}"})

def api_get(path):
    """Бір GET сұраныс. 404 болса None қайтарады."""
    url = path if path.startswith("http") else BASE_URL + path
    for attempt in range(5):
        time.sleep(0.2)
        try:
            r = session.get(url, timeout=60)
        except requests.RequestException as e:
            print(f"Байланыс қатесі ({attempt+1}/5): {e}")
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 404:
            return None
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 ** attempt)
            continue
        if r.status_code in (401, 403):
            raise Exception(f"{r.status_code}: токен жарамсыз немесе рұқсат жоқ")
        r.raise_for_status()
        return r.json()
    raise Exception(f"5 рет сәтсіз: {url}")

# Тексеру
test_pid = next(iter(unresolved_pids))
print("Тексерілетін ID:", test_pid)

for path in [f"/v3/plans/view/{test_pid}", f"/v3/plans/{test_pid}"]:
    try:
        r = session.get(BASE_URL + path, timeout=60)
        print(f"\n{path} -> {r.status_code}")
        print(r.text[:1500])
    except Exception as e:
        print(path, "қате:", e)

Тексерілетін ID: 79691777

/v3/plans/view/79691777 -> 200
{"id":79691777,"rootrecord_id":74970880,"idc":"","sys_organizator_id":12562,"sys_subjects_id":12562,"subject_biin":"990740002243","subject_name_ru":"РГП \"Больница МЦ УДП РК\"","subject_name_kz":"\"ҚР ПІБ МО ауруханасы\" РМК","name_ru":"Криопреципитат","name_kz":"Криопреципитат","ref_trade_methods_id":3,"ref_enstru_id":997668,"ref_units_code":"639","code_gu":"","count":35,"price":81170,"amount":2840950,"ref_months_id":7,"ref_pln_point_status_id":320,"pln_point_year":2025,"ref_subject_type_id":1,"ref_enstru_code":"212021.200.000017","ref_finsource_id":0,"ref_abp_code":0,"is_qvazi":0,"date_create":"2025-07-14 20:01:19","timestamp":"2025-11-03 17:39:03","ref_point_type_id":1,"desc_ru":"замороженный","desc_kz":"мұздатылған","extra_desc_kz":"","extra_desc_ru":"Криопреципитат","sum_1":2840950,"sum_2":0,"sum_3":0,"supply_date_ru":"По заявке заказчика в течение 16 календарных дней","prepayment":0,"ref_justification_id":0,"ref_amendment_

In [ ]:
import json, shutil
from concurrent.futures import ThreadPoolExecutor

def save(entity, obj_id, payload, bin_=None, year=None):
    db.execute("INSERT OR REPLACE INTO raw VALUES (?,?,?,?,?)",
               (entity, str(obj_id), bin_, year, json.dumps(payload, ensure_ascii=False)))

def exists(entity, obj_id):
    return db.execute("SELECT 1 FROM raw WHERE entity=? AND obj_id=?",
                      (entity, str(obj_id))).fetchone() is not None

def safe_get(path):
    """Қате болса False қайтарады (кейін қайта жүктеледі), 404 болса None."""
    try:
        return api_get(path)
    except Exception as e:
        print(f"   қате {path}: {e}")
        return False

# Тізімді Drive-қа сақтаймыз
with open(f"{SAVE_DIR}/missing_plan_ids.json", "w") as f:
    json.dump(sorted(unresolved_pids), f)

todo = [p for p in sorted(unresolved_pids)
        if not exists("plan_hist", p) and not exists("plan_hist_404", p)]
print("Жүктелетін:", len(todo))

t0 = time.time()
for s in range(0, len(todo), 200):
    chunk = todo[s:s+200]
    with ThreadPoolExecutor(4) as ex:
        results = list(ex.map(safe_get, [f"/v3/plans/view/{p}" for p in chunk]))
    for pid, data in zip(chunk, results):
        if data is False:
            continue                                 # қате: келесі іске қосуда қайталанады
        if data and data.get("id"):
            save("plan_hist", pid, data, data.get("subject_biin"),
                 data.get("pln_point_year") or data.get("plan_fin_year"))
        else:
            save("plan_hist_404", pid, {})           # табылмады: қайта сұратпаймыз
    db.commit()
    if (s // 200) % 5 == 4:
        shutil.copy(DB_LOCAL, DB_DRIVE)              # әр 1000 тармақ сайын Drive-қа
    print(f"{min(s+200, len(todo))}/{len(todo)} ({(time.time()-t0)/60:.1f} мин)")

db.commit()
shutil.copy(DB_LOCAL, DB_DRIVE)
print("Дайын.")

Жүктелетін: 10176
200/10176 (0.3 мин)
400/10176 (0.7 мин)
600/10176 (1.0 мин)
800/10176 (1.3 мин)
1000/10176 (1.9 мин)
1200/10176 (2.2 мин)
1400/10176 (2.5 мин)
1600/10176 (2.8 мин)
1800/10176 (3.1 мин)
2000/10176 (3.5 мин)
2200/10176 (3.9 мин)
2400/10176 (4.2 мин)
2600/10176 (4.5 мин)
2800/10176 (4.8 мин)
3000/10176 (5.2 мин)
3200/10176 (5.6 мин)
3400/10176 (5.9 мин)
3600/10176 (6.2 мин)
3800/10176 (6.6 мин)
4000/10176 (7.0 мин)
4200/10176 (7.4 мин)
4400/10176 (7.6 мин)
4600/10176 (7.9 мин)
4800/10176 (8.2 мин)
5000/10176 (8.6 мин)
5200/10176 (8.9 мин)
5400/10176 (9.2 мин)
5600/10176 (9.5 мин)
5800/10176 (9.8 мин)
6000/10176 (10.1 мин)
6200/10176 (10.4 мин)
6400/10176 (10.7 мин)
6600/10176 (11.0 мин)
6800/10176 (11.3 мин)
7000/10176 (11.6 мин)
Байланыс қатесі (1/5): ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
7200/10176 (11.9 мин)
7400/10176 (12.2 мин)
7600/10176 (12.5 мин)
7800/10176 (12.8 мин)
8000/10176 (13.1 мин)
8200/10176 (13.4 ми

In [ ]:
print(q("SELECT entity, COUNT(*) AS n FROM raw WHERE entity LIKE 'plan_hist%' GROUP BY entity"))
print(q("""
SELECT json_extract(payload,'$.is_deleted') AS is_deleted,
       json_extract(payload,'$.is_active')  AS is_active,
       year, COUNT(*) AS n
FROM raw WHERE entity='plan_hist'
GROUP BY 1, 2, 3 ORDER BY n DESC
"""))
print(q("""
SELECT bin, COUNT(*) AS n FROM raw WHERE entity='plan_hist'
GROUP BY bin ORDER BY n DESC LIMIT 10
"""))

          entity     n
0      plan_hist  4633
1  plan_hist_404  5543
   is_deleted  is_active  year     n
0           1          0  2025  2038
1           1          0  2024  2027
2           0          1  2026   298
3           1          0  2026   265
4           0          1  2023     5
            bin     n
0  990740002243  1630
1  030440003698   396
2  971040001050   372
3  231040023028   361
4  020440003656   346
5  100140011059   272
6  981140001551   237
7  051040005150   166
8  940940000384   165
9  980440001034   145


In [ ]:
from collections import Counter

hist_ids = {r[0] for r in db.execute("SELECT obj_id FROM raw WHERE entity='plan_hist'")}

stats = Counter()
unresolved = []
for pid, lid, bin_, year in db.execute("SELECT pid, lot_id, bin, year FROM miss").fetchall():
    if str(pid) in hist_ids:
        stats["plan_hist арқылы"] += 1
    elif any(str(p) in plan_ids for p in lot_points.get(lid, [])):
        stats["лот арқылы"] += 1
    elif lot_enstru.get(lid):
        stats["лоттағы ЕНС ТРУ"] += 1
    else:
        stats["шешілмеген"] += 1
        unresolved.append((pid, lid, bin_, year))

TOTAL = 105166
print(dict(stats))
print(f"Жалпы қамту: {100 * (TOTAL - stats['шешілмеген']) / TOTAL:.1f}%")

# Шешілмегендердің құрылымы
nf = pd.DataFrame(unresolved, columns=["pid", "lot_id", "bin", "year"])
print("\nЖылдар бойынша:\n", nf.groupby("year").size())
print("\nҰйымдар бойынша:\n", nf.groupby("bin").size().sort_values(ascending=False).head(10))
print("\nЛоты жоқ жолдар:", int((nf.lot_id.isna() | (nf.lot_id == 0)).sum()))

{'plan_hist арқылы': 6900, 'лот арқылы': 2752, 'шешілмеген': 5534, 'лоттағы ЕНС ТРУ': 657}
Жалпы қамту: 94.7%

Жылдар бойынша:
 year
2024       3
2025     120
2026    5411
dtype: int64

Ұйымдар бойынша:
 bin
020440003656    1337
970940001378     977
980440001034     527
231040023028     519
051040005150     517
171041003124     383
210240033968     297
780140000023     254
230740013340     184
150540000186     138
dtype: int64

Лоты жоқ жолдар: 289


In [ ]:
nf_ids = [int(r[0]) for r in db.execute("SELECT obj_id FROM raw WHERE entity='plan_hist_404'")]
found_ids = [int(x) for x in hist_ids]

print("Табылғандар ID ауқымы:", min(found_ids), "–", max(found_ids))
print("404 ID ауқымы:        ", min(nf_ids), "–", max(nf_ids))
print("404 ID-лердің медианасы:", int(pd.Series(nf_ids).median()),
      "| табылғандардың медианасы:", int(pd.Series(found_ids).median()))

# 404 болған 10 ID-ді қайта сұратамыз
import random
for pid in random.sample(nf_ids, 10):
    r = session.get(f"{BASE_URL}/v3/plans/view/{pid}", timeout=60)
    print(pid, r.status_code)
    time.sleep(0.3)

Табылғандар ID ауқымы: 55185788 – 87988464
404 ID ауқымы:         57958779 – 88096741
404 ID-лердің медианасы: 86956174 | табылғандардың медианасы: 73135082
87129802 404
81258031 404
86958670 404
87373033 404
81749108 404
86956251 404
87396172 404
85384716 404
86294721 404
87443705 404


In [ ]:
CLEAN_LOCAL = "/content/goszakup_clean.db"
CLEAN_DRIVE = f"{SAVE_DIR}/goszakup_clean.db"
clean = sqlite3.connect(CLEAN_LOCAL)

rows = []
for entity, payload in db.execute(
        "SELECT entity, payload FROM raw WHERE entity IN ('plan', 'plan_hist')"):
    p = json.loads(payload)
    rows.append({
        "plan_id":       p.get("id"),
        "root_id":       p.get("rootrecord_id"),
        "source":        entity,                      # 'plan' = белсенді, 'plan_hist' = ескі нұсқа
        "customer_bin":  p.get("subject_biin"),
        "year":          p.get("pln_point_year") or p.get("plan_fin_year"),
        "month":         p.get("ref_months_id"),
        "enstru_code":   p.get("ref_enstru_code"),
        "unit_code":     p.get("ref_units_code"),
        "name_ru":       p.get("name_ru"),
        "name_kz":       p.get("name_kz"),
        "desc_ru":       p.get("desc_ru"),
        "quantity":      p.get("count"),
        "price":         p.get("price"),
        "amount":        p.get("amount"),
        "trade_method":  p.get("ref_trade_methods_id"),
        "subject_type":  p.get("ref_subject_type_id"),
        "is_deleted":    p.get("is_deleted"),
        "date_create":   p.get("date_create"),
    })

plan_items = pd.DataFrame(rows).drop_duplicates("plan_id")
plan_items.to_sql("plan_items", clean, if_exists="replace", index=False)
clean.execute("CREATE INDEX IF NOT EXISTS ix_plan_items_id ON plan_items(plan_id)")
clean.execute("CREATE INDEX IF NOT EXISTS ix_plan_items_enstru ON plan_items(enstru_code)")
clean.commit()
shutil.copy(CLEAN_LOCAL, CLEAN_DRIVE)

print("Жолдар:", len(plan_items))
print(plan_items.groupby("source").size())
print("\nЕНС ТРУ коды жоқ:", plan_items.enstru_code.isna().sum())
print("\nЕң жиі 15 өлшем бірлігі:\n", plan_items.unit_code.value_counts().head(15))

# Белсенді жоспар тармағында КАТО (өңір) өрісі бар ма?
sample = json.loads(db.execute(
    "SELECT payload FROM raw WHERE entity='plan' LIMIT 1").fetchone()[0])
print("\nКАТО-ға қатысты өрістер:", [k for k in sample if "kato" in k.lower()])

Жолдар: 78733
source
plan         74100
plan_hist     4633
dtype: int64

ЕНС ТРУ коды жоқ: 0

Ең жиі 15 өлшем бірлігі:
 unit_code
796     37980
5114     8010
778      7794
166      7447
704      2390
839      2310
5111     2306
872      2103
112      1377
980      1182
868      1091
006       913
736       693
715       601
881       599
Name: count, dtype: int64

КАТО-ға қатысты өрістер: ['kato']


In [ ]:
r = session.get(f"{BASE_URL}/v3/refs", timeout=60)
print(r.status_code)
print(r.text[:3000])

200
[]


In [ ]:
print(q("""
SELECT COUNT(DISTINCT json_extract(payload,'$.supplier_biin')) AS suppliers
FROM raw WHERE entity='contract'
"""))

r = session.get(f"{BASE_URL}/v3/rnu", timeout=60)
print("\nРНУ:", r.status_code)
print(r.text[:1000])

   suppliers
0       9217

РНУ: 200
{"total":17291,"limit":50,"next_page":"/v3/rnu?page=next&search_after=17242","items":[{"id":17291,"pid":0,"supplier_biin":"150540016043","supplier_innunp":null,"supplier_name_ru":"Товарищество с ограниченной ответственностью \"ТехПромЛидер\"","supplier_name_kz":"\"ТехПромЛидер\" жауапкершілігі шектеулі серіктестігі","kato_list":[0],"system_id":2},{"id":17290,"pid":0,"supplier_biin":"180840008851","supplier_innunp":null,"supplier_name_ru":"Товарищество с ограниченной ответственностью \"ЦентрХим\"","supplier_name_kz":"\"ЦентрХим\" жауапкершілігі шектеулі серіктестігі","kato_list":[0],"system_id":2},{"id":17289,"pid":0,"supplier_biin":"941124400425","supplier_innunp":null,"supplier_name_ru":"ТӨЛЕУОВА БАЛЖАН ДҮЙСЕНБЕКҚЫЗЫ","supplier_name_kz":"ТӨЛЕУОВА БАЛЖАН ДҮЙСЕНБЕКҚЫЗЫ","kato_list":[0],"system_id":2},{"id":17288,"pid":0,"supplier_biin":"870804303729","supplier_innunp":null,"supplier_name_ru":"ИП Ескалиев","supplier_name_kz":"ИП Ескалиев","kato_list":[

In [ ]:
def paginate(path):
    while path:
        data = api_get(path)
        if not data:
            return
        if isinstance(data, list):
            yield from data
            return
        yield from data.get("items") or []
        nxt = data.get("next_page")
        if nxt and not nxt.startswith("/v3/"):
            nxt = "/v3" + (nxt if nxt.startswith("/") else "/" + nxt)
        path = nxt or None

candidates = [
    "ref_units", "ref_trade_methods", "ref_lot_status", "ref_lots_status",
    "ref_buy_status", "ref_pln_point_status", "ref_subject_type", "ref_subject_types",
    "ref_contract_status", "ref_kato", "ref_months", "ref_finsource",
    "ref_type_trade", "ref_point_type", "ref_contract_type", "ref_currency",
]
for name in candidates:
    try:
        r = session.get(f"{BASE_URL}/v3/refs/{name}", timeout=15)
        preview = r.text[:150].replace("\n", " ")
        print(f"{name:24} {r.status_code}  {preview}")
    except requests.RequestException as e:
        print(f"{name:24} ҚАТЕ: {type(e).__name__}")
    time.sleep(0.3)

ref_units                200  {"total":621,"limit":50,"next_page":"/v3/refs/ref_units?page=next&search_after=931","items":[{"code":"980","code2":"7.348","alpha_code":"раб","name_kz
ref_trade_methods        200  {"total":35,"limit":50,"next_page":"","items":[{"code":"202","name_kz":"Толық бітіріп берілетін құрылыс бойынша тендер","name_ru":"Тендер по строитель
ref_lot_status           200  {"error":"Справочник не найден","code":404}
ref_lots_status          200  {"total":37,"limit":50,"next_page":"","items":[{"id":550,"name_ru":"Принятие решение о пересмотре итогов","name_kz":"Принятие решение о пересмотре ито
ref_buy_status           200  {"total":35,"limit":50,"next_page":"","items":[{"id":550,"name_ru":"Принятие решение о пересмотре итогов","name_kz":"Принятие решение о пересмотре ито
ref_pln_point_status     200  {"total":36,"limit":50,"next_page":"","items":[{"id":560,"name_ru":"Перемещение с предварительного ГП","name_kz":"Перемещение с предварительного ГП","
ref_subject_type     

In [ ]:
n = 0
for item in paginate("/v3/rnu"):
    save("rnu", item["id"], item, item.get("supplier_biin"))
    n += 1
    if n % 2000 == 0:
        db.commit()
        print(f"  РНУ: {n}...")
db.commit()
shutil.copy(DB_LOCAL, DB_DRIVE)
print("РНУ жазбалары:", n)

# Біздің жеткізушілердің қаншасы РНУ-да бар?
print(q("""
SELECT COUNT(DISTINCT json_extract(c.payload,'$.supplier_biin')) AS suppliers_in_rnu
FROM raw c
WHERE c.entity='contract'
  AND json_extract(c.payload,'$.supplier_biin') IN (SELECT bin FROM raw WHERE entity='rnu')
"""))

  РНУ: 2000...
  РНУ: 4000...
  РНУ: 6000...
  РНУ: 8000...
  РНУ: 10000...
  РНУ: 12000...
  РНУ: 14000...
  РНУ: 16000...
РНУ жазбалары: 17291
   suppliers_in_rnu
0              2749


In [ ]:
supplier_bins = [r[0] for r in db.execute("""
    SELECT DISTINCT json_extract(payload,'$.supplier_biin') FROM raw
    WHERE entity='contract' AND json_extract(payload,'$.supplier_biin') IS NOT NULL
""")]
all_bins = sorted(set(BINS) | set(supplier_bins))
todo = [b for b in all_bins if not exists("subject", b) and not exists("subject_404", b)]
print("Жүктелетін ұйымдар:", len(todo))

t0 = time.time()
for s in range(0, len(todo), 200):
    chunk = todo[s:s+200]
    with ThreadPoolExecutor(4) as ex:
        results = list(ex.map(safe_get, [f"/v3/subject/biin/{b}" for b in chunk]))
    for b, data in zip(chunk, results):
        if data is False:
            continue                     # қате: келесі іске қосуда қайталанады
        if data:
            save("subject", b, data, b)
        else:
            save("subject_404", b, {})
    db.commit()
    if (s // 200) % 5 == 4:
        shutil.copy(DB_LOCAL, DB_DRIVE)
    print(f"{min(s+200, len(todo))}/{len(todo)} ({(time.time()-t0)/60:.1f} мин)")

db.commit()
shutil.copy(DB_LOCAL, DB_DRIVE)
print("Дайын.")

Жүктелетін ұйымдар: 9230
200/9230 (0.3 мин)
400/9230 (0.6 мин)
600/9230 (0.9 мин)
800/9230 (1.2 мин)
1000/9230 (1.7 мин)
1200/9230 (2.0 мин)
1400/9230 (2.3 мин)
1600/9230 (2.6 мин)
1800/9230 (2.9 мин)
2000/9230 (3.4 мин)
2200/9230 (3.7 мин)
2400/9230 (4.0 мин)
2600/9230 (4.3 мин)
2800/9230 (4.6 мин)
3000/9230 (5.0 мин)
3200/9230 (5.4 мин)
3400/9230 (5.7 мин)
3600/9230 (6.0 мин)
3800/9230 (6.3 мин)
4000/9230 (6.7 мин)
4200/9230 (7.0 мин)
4400/9230 (7.3 мин)
4600/9230 (7.6 мин)
4800/9230 (7.9 мин)
5000/9230 (8.4 мин)
Байланыс қатесі (1/5): HTTPSConnectionPool(host='ows.goszakup.gov.kz', port=443): Read timed out. (read timeout=60)
5200/9230 (9.4 мин)
5400/9230 (9.7 мин)
5600/9230 (10.1 мин)
5800/9230 (10.4 мин)
6000/9230 (10.8 мин)
6200/9230 (11.1 мин)
6400/9230 (11.4 мин)
Байланыс қатесі (1/5): HTTPSConnectionPool(host='ows.goszakup.gov.kz', port=443): Read timed out. (read timeout=60)
Байланыс қатесі (1/5): HTTPSConnectionPool(host='ows.goszakup.gov.kz', port=443): Read timed out. (rea

In [ ]:
def paginate_safe(path, label="", max_pages=1000):
    """Қайталанатын next_page-тен қорғалған және прогресс көрсететін беттеу."""
    seen, page = set(), 0
    while path and path not in seen and page < max_pages:
        seen.add(path)
        page += 1
        data = api_get(path)
        if not data or not isinstance(data, dict):
            return
        yield from data.get("items") or []
        if page % 20 == 0:
            print(f"    {label}: {page} бет...")
        nxt = data.get("next_page")
        if nxt and not nxt.startswith("/v3/"):
            nxt = "/v3" + (nxt if nxt.startswith("/") else "/" + nxt)
        path = nxt or None
    if path in seen:
        print(f"    {label}: next_page қайталанды, тоқтатылды")

db.commit()   # тоқтатқанға дейін жүктелгендерін сақтаймыз

REFS = ["ref_units", "ref_trade_methods", "ref_lots_status", "ref_buy_status",
        "ref_pln_point_status", "ref_subject_type", "ref_contract_status",
        "ref_months", "ref_finsource", "ref_type_trade", "ref_point_type",
        "ref_contract_type", "ref_currency", "ref_kato"]   # ең үлкені соңында

for name in REFS:
    if exists("ref", name):
        print(f"{name:24} бұрын жүктелген")
        continue
    t0 = time.time()
    try:
        items = list(paginate_safe(f"/v3/refs/{name}", label=name))
    except Exception as e:
        print(f"{name:24} ҚАТЕ: {e}")
        continue
    if items:
        save("ref", name, items)
        db.commit()
        print(f"{name:24} {len(items)} жазба ({time.time()-t0:.0f} с)")
    else:
        print(f"{name:24} БОС")

shutil.copy(DB_LOCAL, DB_DRIVE)

ref_units                бұрын жүктелген
ref_trade_methods        бұрын жүктелген
ref_lots_status          бұрын жүктелген
ref_buy_status           бұрын жүктелген
ref_pln_point_status     бұрын жүктелген
ref_subject_type         бұрын жүктелген
ref_contract_status      бұрын жүктелген
ref_months               бұрын жүктелген
ref_finsource            бұрын жүктелген
ref_type_trade           бұрын жүктелген
ref_point_type           бұрын жүктелген
ref_contract_type        бұрын жүктелген
    ref_currency: next_page қайталанды, тоқтатылды
ref_currency             100 жазба (1 с)
ref_kato                 бұрын жүктелген


'/content/drive/MyDrive/Goszakup_agent/goszakup_raw.db'

In [ ]:
rnu_bins = [r[0] for r in db.execute("""
    SELECT DISTINCT json_extract(c.payload,'$.supplier_biin')
    FROM raw c
    WHERE c.entity='contract'
      AND json_extract(c.payload,'$.supplier_biin') IN (SELECT bin FROM raw WHERE entity='rnu')
""")]
todo = [b for b in rnu_bins if not exists("rnu_detail", b) and not exists("rnu_detail_404", b)]
print("Жүктелетін:", len(todo))

t0 = time.time()
for s in range(0, len(todo), 200):
    chunk = todo[s:s+200]
    with ThreadPoolExecutor(3) as ex:
        results = list(ex.map(safe_get, [f"/v3/rnu/{b}" for b in chunk]))
    for b, data in zip(chunk, results):
        if data is False:
            continue
        items = (data or {}).get("items") or []
        if items:
            save("rnu_detail", b, items, b)
        else:
            save("rnu_detail_404", b, {})
    db.commit()
    print(f"{min(s+200, len(todo))}/{len(todo)} ({(time.time()-t0)/60:.1f} мин)")

shutil.copy(DB_LOCAL, DB_DRIVE)
print("Дайын.")

Жүктелетін: 2749
200/2749 (0.4 мин)
400/2749 (0.8 мин)
600/2749 (1.2 мин)
800/2749 (1.6 мин)
1000/2749 (2.0 мин)
1200/2749 (2.9 мин)
1400/2749 (3.3 мин)
1600/2749 (3.8 мин)
1800/2749 (4.2 мин)
2000/2749 (4.7 мин)
2200/2749 (5.1 мин)
Байланыс қатесі (1/5): HTTPSConnectionPool(host='ows.goszakup.gov.kz', port=443): Read timed out. (read timeout=60)
Байланыс қатесі (1/5): HTTPSConnectionPool(host='ows.goszakup.gov.kz', port=443): Read timed out. (read timeout=60)
Байланыс қатесі (1/5): HTTPSConnectionPool(host='ows.goszakup.gov.kz', port=443): Read timed out. (read timeout=60)
2400/2749 (7.1 мин)
2600/2749 (7.5 мин)
2749/2749 (7.8 мин)
Дайын.


In [ ]:
print(q("""
SELECT entity, COUNT(*) AS n FROM raw
WHERE entity IN ('subject', 'subject_404', 'rnu', 'rnu_detail', 'rnu_detail_404', 'ref')
GROUP BY entity
"""))

all_bins = set(BINS) | {r[0] for r in db.execute("""
    SELECT DISTINCT json_extract(payload,'$.supplier_biin') FROM raw
    WHERE entity='contract' AND json_extract(payload,'$.supplier_biin') IS NOT NULL""")}
loaded = {r[0] for r in db.execute(
    "SELECT obj_id FROM raw WHERE entity IN ('subject', 'subject_404')")}
missing = all_bins - loaded
print("\nЖүктелмеген ұйымдар:", len(missing))
print("27 ұйымның бәрі бар ма:", all(b in loaded for b in BINS))

           entity      n
0             ref     14
1             rnu  17291
2      rnu_detail   2636
3  rnu_detail_404    113
4         subject   8974
5     subject_404    256

Жүктелмеген ұйымдар: 0
27 ұйымның бәрі бар ма: True


In [ ]:
# 1) Шикі дерекқордағы барлық объектілер
print(q("SELECT entity, COUNT(*) AS n FROM raw GROUP BY entity ORDER BY n DESC").to_string())

# 2) Анықтамалықтар және олардағы жазбалар саны
print("\nАнықтамалықтар:")
for name, payload in db.execute("SELECT obj_id, payload FROM raw WHERE entity='ref' ORDER BY obj_id"):
    print(f"  {name:24} {len(json.loads(payload))}")

# 3) Ұйымдар толық жүктелді ме
all_bins = set(BINS) | {r[0] for r in db.execute("""
    SELECT DISTINCT json_extract(payload,'$.supplier_biin') FROM raw
    WHERE entity='contract' AND json_extract(payload,'$.supplier_biin') IS NOT NULL""")}
loaded = {r[0] for r in db.execute(
    "SELECT obj_id FROM raw WHERE entity IN ('subject', 'subject_404')")}
print("\nЖүктелмеген ұйымдар:", len(all_bins - loaded))
print("27 ұйымның бәрі бар ма:", all(b in loaded for b in BINS))

# 4) РНУ мәліметінің бір мысалы (күндері бар ма)
row = db.execute("SELECT payload FROM raw WHERE entity='rnu_detail' LIMIT 1").fetchone()
if row:
    item = json.loads(row[0])[0]
    print("\nРНУ мысалы:", {k: item.get(k) for k in
          ["supplier_biin", "start_date", "end_date", "ref_reason_id", "system_id", "customer_biin"]})

# 5) Drive-қа соңғы көшірме
db.commit()
shutil.copy(DB_LOCAL, DB_DRIVE)
print("\nDrive-қа сақталды.")

            entity       n
0              lot  116368
1    contract_unit  105166
2             plan   74100
3     trd_buy_skip   38045
4         contract   37039
5          trd_buy   36488
6              rnu   17291
7          subject    8974
8    plan_hist_404    5543
9        plan_hist    4633
10      rnu_detail    2636
11     subject_404     256
12  rnu_detail_404     113
13            done      27
14             ref      14

Анықтамалықтар:
  ref_buy_status           35
  ref_contract_status      33
  ref_contract_type        2
  ref_currency             100
  ref_finsource            9
  ref_kato                 16844
  ref_lots_status          37
  ref_months               13
  ref_pln_point_status     36
  ref_point_type           3
  ref_subject_type         3
  ref_trade_methods        35
  ref_type_trade           73
  ref_units                1399

Жүктелмеген ұйымдар: 0
27 ұйымның бәрі бар ма: True

РНУ мысалы: {'supplier_biin': '000102500374', 'start_date': '2026-04-17 00:

In [ ]:
import os

# Runtime қайта басталған болса, таза дерекқорды Drive-тан аламыз
if not os.path.exists(CLEAN_LOCAL) and os.path.exists(CLEAN_DRIVE):
    shutil.copy(CLEAN_DRIVE, CLEAN_LOCAL)
clean = sqlite3.connect(CLEAN_LOCAL)

for name, payload in db.execute("SELECT obj_id, payload FROM raw WHERE entity='ref'").fetchall():
    df = pd.DataFrame(json.loads(payload))

    # SQLite тізім мен сөздікті сақтай алмайды, сондықтан оларды JSON мәтінге айналдырамыз
    for col in df.columns:
        if df[col].map(lambda v: isinstance(v, (list, dict))).any():
            df[col] = df[col].map(
                lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v)

    key = "id" if "id" in df.columns else ("code" if "code" in df.columns else None)
    before = len(df)
    df = df.drop_duplicates(subset=[key] if key else None)
    df.to_sql(name, clean, if_exists="replace", index=False)
    print(f"{name:24} {before:>6} -> {len(df):>6}  (кілт: {key})")

clean.commit()
shutil.copy(CLEAN_LOCAL, CLEAN_DRIVE)

# Біздің деректердегі өлшем бірліктерінің бәрі анықтамалықта бар ма?
units = pd.read_sql("SELECT * FROM ref_units", clean)
print("\nref_units бағандары:", units.columns.tolist())
print(units.head(3).to_string())

used = pd.read_sql("SELECT unit_code, COUNT(*) AS n FROM plan_items GROUP BY unit_code", clean)
known = set(units["code"].astype(str))
used["found"] = used["unit_code"].astype(str).isin(known)
print(f"\nҚолданылған кодтар: {len(used)}, анықтамалықта жоқ: {(~used.found).sum()}")
print(used[~used.found].sort_values("n", ascending=False).head(10).to_string())

# КАТО анықтамалығының құрылымы
kato = pd.read_sql("SELECT * FROM ref_kato LIMIT 3", clean)
print("\nref_kato бағандары:", kato.columns.tolist())
print(kato.to_string())

ref_buy_status               35 ->     35  (кілт: id)
ref_contract_status          33 ->     33  (кілт: id)
ref_contract_type             2 ->      2  (кілт: id)
ref_currency                100 ->     50  (кілт: code)
ref_finsource                 9 ->      9  (кілт: id)
ref_kato                  16844 ->  16844  (кілт: code)
ref_lots_status              37 ->     37  (кілт: id)
ref_months                   13 ->     13  (кілт: id)
ref_pln_point_status         36 ->     36  (кілт: id)
ref_point_type                3 ->      3  (кілт: id)
ref_subject_type              3 ->      3  (кілт: id)
ref_trade_methods            35 ->     35  (кілт: id)
ref_type_trade               73 ->     73  (кілт: id)
ref_units                  1399 ->    621  (кілт: code)

ref_units бағандары: ['code', 'code2', 'alpha_code', 'name_kz', 'name_ru']
  code  code2       alpha_code                   name_kz                       name_ru
0  980  7.348              раб                     Жұмыс                   

In [ ]:
def raw_df(entity):
    """Шикі дерекқордағы бір түрдің барлық JSON-ын DataFrame-ге айналдырады."""
    rows = [json.loads(p) for (p,) in db.execute(
        "SELECT payload FROM raw WHERE entity=?", (entity,))]
    return pd.DataFrame(rows)

def ref_map(table, key="id", col="name_ru"):
    """Анықтамалықтан {код: атауы} сөздігін жасайды."""
    df = pd.read_sql(f"SELECT {key}, {col} FROM {table}", clean)
    return dict(zip(df[key].astype(str), df[col]))

def to_region(kato):
    """КАТО кодынан облыс кодын алады: 791910000 -> 790000000."""
    s = str(kato or "")
    return s[:2] + "0000000" if len(s) == 9 and s[:2] != "00" else None

In [ ]:
regions = pd.read_sql("""
    SELECT code AS region_code, name_ru, name_kz FROM ref_kato
    WHERE code LIKE '__0000000'
""", clean)
regions.to_sql("regions", clean, if_exists="replace", index=False)
print("Өңірлер:", len(regions))

rows = []
for (p,) in db.execute("SELECT payload FROM raw WHERE entity IN ('plan', 'plan_hist')"):
    item = json.loads(p)
    for k in item.get("kato") or []:
        rows.append({
            "plan_id":      item["id"],
            "kato_code":    k.get("ref_kato_code"),
            "region_code":  to_region(k.get("ref_kato_code")),
            "quantity":     k.get("count"),
            "address_ru":   k.get("full_delivery_place_name_ru"),
        })
plan_delivery = pd.DataFrame(rows)
plan_delivery.to_sql("plan_delivery", clean, if_exists="replace", index=False)
clean.execute("CREATE INDEX IF NOT EXISTS ix_pd_plan ON plan_delivery(plan_id)")
print("Жеткізу орындары:", len(plan_delivery),
      "| жоспар тармақтары:", plan_delivery.plan_id.nunique())

Өңірлер: 21
Жеткізу орындары: 28041 | жоспар тармақтары: 27974


In [ ]:
tb = raw_df("trd_buy")
methods   = ref_map("ref_trade_methods")
buy_stat  = ref_map("ref_buy_status")
subj_type = ref_map("ref_subject_type")

announcements = pd.DataFrame({
    "trd_buy_id":       tb["id"],
    "number_anno":      tb["number_anno"],
    "name_ru":          tb["name_ru"],
    "name_kz":          tb["name_kz"],
    "customer_bin":     tb["customer_bin"],
    "org_bin":          tb["org_bin"],
    "total_sum":        tb["total_sum"],
    "count_lots":       tb["count_lots"],
    "trade_method_id":  tb["ref_trade_methods_id"],
    "trade_method":     tb["ref_trade_methods_id"].astype(str).map(methods),
    "subject_type":     tb["ref_subject_type_id"].astype(str).map(subj_type),
    "status":           tb["ref_buy_status_id"].astype(str).map(buy_stat),
    "is_repeat":        (tb["parent_id"].fillna(0) > 0).astype(int),
    "parent_id":        tb["parent_id"],
    "publish_date":     tb["publish_date"],
    "end_date":         tb["end_date"],
    "itogi_date":       tb["itogi_date_public"],
    "year":             tb["publish_date"].str[:4].astype("Int64"),
})
announcements.to_sql("announcements", clean, if_exists="replace", index=False)
clean.execute("CREATE INDEX IF NOT EXISTS ix_ann_id ON announcements(trd_buy_id)")
print("\nХабарландырулар:", len(announcements))
print(announcements.trade_method.value_counts().head(8))


Хабарландырулар: 36488
trade_method
Запрос ценовых предложений                                    21344
Из одного источника по несостоявшимся закупкам                10666
Открытый конкурс                                               3920
Конкурс с предварительным квалификационным отбором              185
Второй этап конкурса с использованием рамочного соглашения      127
Аукцион                                                          81
Аукцион (с 2022)                                                 78
Конкурс с использованием рейтингово-балльной системы             49
Name: count, dtype: int64


In [ ]:
lt = raw_df("lot")
lot_stat = ref_map("ref_lots_status")

def first_kato(lst):
    lst = [k for k in (lst or []) if k and str(k) != "0"]
    return str(lst[0]) if lst else None

lots = pd.DataFrame({
    "lot_id":        lt["id"],
    "lot_number":    lt["lot_number"],
    "trd_buy_id":    lt["trd_buy_id"],
    "customer_bin":  lt["customer_bin"],
    "name_ru":       lt["name_ru"],
    "name_kz":       lt["name_kz"],
    "description_ru": lt["description_ru"],
    "quantity":      lt["count"],
    "amount":        lt["amount"],
    "is_union":      lt["union_lots"],
    "status":        lt["ref_lot_status_id"].astype(str).map(lot_stat),
    "trade_method":  lt["ref_trade_methods_id"].astype(str).map(methods),
    "dumping":       lt["dumping"],
    "kato_code":     lt["pln_point_kato_list"].map(first_kato),
    "point_list":    lt["point_list"].map(json.dumps),
})
lots["region_code"] = lots["kato_code"].map(to_region)
ok = (lots.quantity > 0) & (lots.is_union == 0)
lots.loc[ok, "unit_price"] = lots.loc[ok, "amount"] / lots.loc[ok, "quantity"]

lots.to_sql("lots", clean, if_exists="replace", index=False)
clean.execute("CREATE INDEX IF NOT EXISTS ix_lots_id ON lots(lot_id)")
print("\nЛоттар:", len(lots),
      "| бірлік бағасы бар:", lots.unit_price.notna().sum(),
      "| өңірі бар:", lots.region_code.notna().sum())
print(lots.status.value_counts().head(8))


Лоттар: 116368 | бірлік бағасы бар: 112183 | өңірі бар: 36815
status
Закупка не состоялась                      64783
Закупка состоялась                         46277
Отказ от лота                               4015
Изменена документация                        611
Отменен                                      351
Опубликован (прием ценовых предложений)      183
Опубликован (прием заявок)                    84
Рассмотрение заявок                           32
Name: count, dtype: int64


In [ ]:
ct = raw_df("contract")
c_stat = ref_map("ref_contract_status")
c_type = ref_map("ref_contract_type")

contracts = pd.DataFrame({
    "contract_id":       ct["id"],
    "root_id":           ct["root_id"],
    "parent_id":         ct["parent_id"],
    "contract_number":   ct.get("contract_number_sys", ct.get("contract_number")),
    "trd_buy_id":        ct["trd_buy_id"],
    "customer_bin":      ct["customer_bin"],
    "supplier_bin":      ct["supplier_biin"],
    "sum_no_vat":        ct.get("contract_sum"),
    "sum_with_vat":      ct.get("contract_sum_wnds"),
    "fact_sum_with_vat": ct.get("fakt_sum_wnds"),
    "sign_date":         ct.get("sign_date"),
    "crdate":            ct["crdate"],
    "fin_year":          ct.get("fin_year"),
    "status":            ct["ref_contract_status_id"].astype(str).map(c_stat),
    "contract_type":     ct.get("ref_contract_type_id", pd.Series(dtype=str)).astype(str).map(c_type),
    "fact_trade_method": ct.get("fakt_trade_methods_id", pd.Series(dtype=str)).astype(str).map(methods),
    "plan_exec_date":    ct.get("plan_exec_date"),
    "fact_exec_date":    ct.get("fakt_exec_date"),
    "end_date":          ct.get("contract_end_date"),
    "deleted":           ct["deleted"],
})
contracts["is_single_source"] = (contracts.trd_buy_id.fillna(0) == 0).astype(int)
contracts["year"] = contracts["crdate"].str[:4].astype("Int64")

# Әр root_id бойынша ең соңғы (ID-і ең үлкен) өшірілмеген нұсқа
live = contracts[contracts.deleted == 0]
latest_ids = set(live.sort_values("contract_id").groupby("root_id").tail(1).contract_id)
contracts["is_latest"] = contracts.contract_id.isin(latest_ids).astype(int)

contracts.to_sql("contracts", clean, if_exists="replace", index=False)
clean.execute("CREATE INDEX IF NOT EXISTS ix_contracts_id ON contracts(contract_id)")
clean.commit()
shutil.copy(CLEAN_LOCAL, CLEAN_DRIVE)

print("\nШарттар:", len(contracts),
      "| соңғы нұсқалар:", contracts.is_latest.sum(),
      "| өшірілгендер:", (contracts.deleted == 1).sum())
print("Бір көзден:", contracts.is_single_source.sum(),
      f"({100 * contracts.is_single_source.mean():.1f}%)")
print("Бос өрістер:", contracts[["sum_no_vat", "sign_date", "status"]].isna().sum().to_dict())
print(contracts.status.value_counts().head(8))


Шарттар: 37039 | соңғы нұсқалар: 30826 | өшірілгендер: 0
Бір көзден: 3747 (10.1%)
Бос өрістер: {'sum_no_vat': 0, 'sign_date': 737, 'status': 0}
status
Исполнен                              22481
Изменен                                6226
Действует                              4309
Расторгнут в одностороннем порядке     1432
Частично исполнен                       899
Не заключен                             730
Расторгнут по соглашению сторон         673
Передан.Действует                       233
Name: count, dtype: int64
